<a href="https://colab.research.google.com/github/zayminhtike/zayminhtike.com/blob/main/Zay_BAN482_Model_Evaluation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Validation Set Approach

In [ ]:
# Import the matplotlib library for plotting visualizations
import matplotlib.pyplot as plt
# Import the pandas library for structured data manipulation using DataFrames
import pandas as pd
# Import the train_test_split utility from scikit-learn to divide datasets into subsets
from sklearn.model_selection import train_test_split

In [ ]:
# Load the bad drivers CSV dataset into a pandas DataFrame
badDrivers = pd.read_csv('bad-drivers.csv')

In [ ]:
# Display the bad drivers data
badDrivers.head()

In [ ]:
# Define the proportions for splitting the dataset into training, validation, and testing sets
trainingProportion = 0.70
validationProportion = 0.10
testProportion = 0.20

# Split off the test dataset (20% of the original sample data) and keep the remaining 80%
trainingAndValidationData, testData = train_test_split(
    badDrivers, test_size=testProportion
)

# Split the remaining 80% of data into training and validation subsets according to their proportional ratio
trainingData, validationData = train_test_split(
    trainingAndValidationData,
    train_size=trainingProportion / (trainingProportion + validationProportion),
)

In [ ]:
# Display training data
trainingData.head()

In [ ]:
# Display validation data
validationData.head()

In [ ]:
# Display test data
testData.head()

In [ ]:
# Display the scatter plot for the entire sample data
plt.scatter(
    badDrivers[['Losses incurred by insurance companies for collisions per insured driver ($)']],
    badDrivers[['Car Insurance Premiums ($)']],
)
plt.xlabel('Losses incurred by insurance companies', fontsize=14)
plt.ylabel('Car insurance premiums', fontsize=14)
plt.xlim(80, 200)
plt.ylim(600, 1400)
plt.title('Sample data')
plt.show()

In [ ]:
# Display the scatter plot for the training data
plt.scatter(
    trainingData[['Losses incurred by insurance companies for collisions per insured driver ($)']],
    trainingData[['Car Insurance Premiums ($)']],
)
plt.xlabel('Losses incurred by insurance companies', fontsize=14)
plt.ylabel('Car insurance premiums', fontsize=14)
plt.xlim(80, 200)
plt.ylim(600, 1400)
plt.title('Training data')
plt.show()

In [ ]:
# Display the scatter plot for the validation data
plt.scatter(
    validationData[['Losses incurred by insurance companies for collisions per insured driver ($)']],
    validationData[['Car Insurance Premiums ($)']],
)
plt.xlabel('Losses incurred by insurance companies', fontsize=14)
plt.ylabel('Car insurance premiums', fontsize=14)
plt.xlim(80, 200)
plt.ylim(600, 1400)
plt.title('Validation data')
plt.show()

In [ ]:
# Display the scatter plot for the test data
plt.scatter(
    testData[['Losses incurred by insurance companies for collisions per insured driver ($)']],
    testData[['Car Insurance Premiums ($)']],
)
plt.xlabel('Losses incurred by insurance companies', fontsize=14)
plt.ylabel('Car insurance premiums', fontsize=14)
plt.xlim(80, 200)
plt.ylim(600, 1400)
plt.title('Test data')
plt.show()

## K-Fold Cross-Validation

In [ ]:
# Import matplotlib for data visualization and plotting
import matplotlib.pyplot as plt
# Import pandas for handling and exploring structured datasets
import pandas as pd
# Import numpy for high-performance mathematical operations on numerical arrays
import numpy as np
# Import linear_model module from sklearn for machine learning linear algorithms
from sklearn import linear_model
# Import the LinearRegression estimator to build standard linear regression models
from sklearn.linear_model import LinearRegression
# Import model evaluation helpers: train_test_split and cross_val_score for evaluating model performance
from sklearn.model_selection import train_test_split, cross_val_score

In [ ]:
# Read the raw bad drivers CSV file into a pandas DataFrame
badDrivers = pd.read_csv('bad-drivers.csv')

# Isolate 20% of the dataset to act as an independent final test set
badDriversTrainingdata, testData = train_test_split(badDrivers, test_size=0.20)



In [ ]:
# --- Setting up the Scikit Learn Workflow -----
## DEFINE
# Store the target feature (insurance premium) and predictor feature (insurance company losses)
# Reshape them into 2D arrays because scikit-learn estimators expect 2D array inputs for features
X = badDriversTrainingdata[
    ['Losses incurred by insurance companies for collisions per insured driver ($)']
    ].values.reshape(-1, 1)
y = badDriversTrainingdata[['Car Insurance Premiums ($)']].values.reshape(-1, 1)

In [ ]:
badDriversTrainingdata.head()

In [ ]:
testData.head()

In [ ]:
## INITIALIZE
# Initialize and fit the ordinary least squares linear regression model on the training features
linModel = LinearRegression()



In [ ]:
## FIT, PREDICT, and EVALUATE using 10-Fold CV

# Perform 10-fold cross-validation on our regression model.
# Since scikit-learn's scorers maximize utility, they return negative MSE scores.
# We apply a minus sign (-) to convert these negative scores back to positive MSE values.
ten_fold_scores = -cross_val_score(
    linModel, X, y, scoring='neg_mean_squared_error', cv=10
)
# Output the computed MSE score for each of the 10 validation folds
ten_fold_scores

In [ ]:
## FIT, PREDICT, and EVALUATE using LOOCV

# Perform Leave-One-Out Cross-Validation (LOOCV) on our dataset.
# Since we have 40 samples in our training subset, setting cv=40 means each sample is used as a validation set once.
# We prepend the minus sign (-) to obtain positive Mean Squared Error scores.
LOOCV_scores = -cross_val_score(linModel, X, y, scoring='neg_mean_squared_error', cv=40)
# Output the computed validation MSE for each of the 40 individual data splits
LOOCV_scores

In [ ]:
# Plot a comparison of the validation errors between 10-fold and LOOCV strategies
# Plot the 10-fold cross-validation scores vertically at x-coordinate 0
plt.plot(np.zeros_like(ten_fold_scores), ten_fold_scores, '.')
# Plot the LOOCV scores vertically at x-coordinate 1
plt.plot(np.zeros_like(LOOCV_scores) + 1, LOOCV_scores, '.')
# Configure the y-axis label and map the discrete x-ticks to their respective approach labels
plt.ylabel('Mean squared errors', fontsize=14);
plt.xticks([0, 1], ['10-fold', 'LOOCV']);

## Bootstrapping

In [ ]:
# Import matplotlib.pyplot library for creating visualizations and plots
import matplotlib.pyplot as plt
# Import pandas library for data manipulation and analysis with DataFrames
import pandas as pd
# Import numpy library for numerical operations and array handling
import numpy as np
# Import resample function from sklearn.utils for bootstrap sampling with replacement
from sklearn.utils import resample
# Import LinearRegression class from sklearn for fitting linear regression models
from sklearn.linear_model import LinearRegression
# Import mean_squared_error function from sklearn for calculating model prediction errors
from sklearn.metrics import mean_squared_error

In [ ]:
# Load data set
badDrivers = pd.read_csv('bad-drivers.csv')

In [ ]:
badDrivers.info()

In [ ]:
# Create bootstrap samples and collect errors

# Initializes an empty list to store the mean squared errors
# calculated in each bootstrap iteration.
bootstrapErrors = []

for i in range(0, 30):
    # Creates a bootstrap sample (boot) by randomly sampling 51 rows from the
    # badDrivers DataFrame with replacement. This means some rows may be
    # selected multiple times, and others may not be selected at all.
    boot = resample(badDrivers, replace=True, n_samples=51)

    # Creates the out-of-bag sample (oob) which consists of the rows from the
    # original badDrivers DataFrame that were not included in the current
    # bootstrap sample.
    oob = badDrivers[~badDrivers.index.isin(boot.index)]

    # Fit a linear model to the bootstrap sample:
    # These lines prepare the data from the bootstrap sample (boot) for the
    # linear regression model. The Losses incurred... column is used as the
    # independent variable (XBoot), and Car Insurance Premiums ($) is used as
    # the dependent variable (yBoot). A LinearRegression model is initialized
    # and then fitted (trained) using this bootstrap sample data.
    XBoot = boot[
        ['Losses incurred by insurance companies for collisions per insured driver ($)']
    ].values.reshape(-1, 1)
    yBoot = boot[['Car Insurance Premiums ($)']].values.reshape(-1, 1)
    linModel = LinearRegression()
    linModel.fit(XBoot, yBoot)

    # Predict y values for the out-of-bag sample
    # These lines prepare the data from the out-of-bag sample (oob) and use the
    # trained linear regression model (linModel) to predict the Car Insurance
    # Premiums ($) (YOobPredicted) based on the Losses incurred... values (XOob)
    # from the out-of-bag sample.
    XOob = oob[
        ['Losses incurred by insurance companies for collisions per insured driver ($)']
    ].values.reshape(-1, 1)
    YOob = oob[['Car Insurance Premiums ($)']].values.reshape(-1, 1)
    YOobPredicted = linModel.predict(XOob)

    # Calculate the error
    bootError = mean_squared_error(YOob, YOobPredicted)
    bootstrapErrors.append(bootError)

In [ ]:
# Calculate the mean of the errors
np.mean(bootstrapErrors)

In [ ]:
# Calculate the standard deviation of the errors
np.std(bootstrapErrors)

In [ ]:
# Plot the errors
plt.plot(bootstrapErrors, np.zeros_like(bootstrapErrors), '.')
# This is the core plotting command.
# bootstrapErrors: This is the list containing the mean squared errors calculated
# during each bootstrap iteration. These values will be plotted on the x-axis.
# np.zeros_like(bootstrapErrors): This creates an array of zeros with the same
# shape as bootstrapErrors. This effectively sets all the y-coordinates to zero,
# causing the data points to be plotted along the x-axis.
# .: This specifies that the data points should be plotted as dots.

plt.xlabel('Bootstrap errors (MSE)', fontsize=14)
# This sets the label for the x-axis to 'Bootstrap errors (MSE)' and increases
# the font size to 14 for better readability.
plt.gca().axes.yaxis.set_ticks([]);
# This line removes the tick marks from the y-axis. Since all the data points
# are plotted on the x-axis (y=0), the y-axis doesn't convey any meaningful
# information, so removing the ticks cleans up the plot.